# Link domains v1 — what kind of site each pre-accusation link points to (AFT variable #16)

Every link the 998 pilot accounts posted before their first accusation: the URL of a link post, and any URL written in a
post's text or a comment. Each link gets one `link_type`:

- **rules first**, by host: `reddit` (links to Reddit itself, `redd.it` short links, relative `/r/…` links), `visual`
  (the agreed visual-media hosts: i./v./preview.redd.it, Reddit galleries, reddituploads, redditmedia images, imgur,
  gfycat, redgifs, giphy, streamable, YouTube), `messaging` (Discord, Telegram, WhatsApp invites);
- **every other registrable domain** was labelled by agents (`link_domain_llm_labels_v1.csv`, guide
  `link_domain_classification_guide.md`): porn, crypto, shop, blog_channel, news, media_host, messaging, social, other;
- `OVERRIDES` below records corrections to agent labels.

`media_host` (image/video hosts outside the agreed visual list) is kept as its own code here; whether it is counted with
`visual` is decided when the variables are built. Output (never overwritten) in `link_domain_output/`.

In [ ]:
# Libraries.
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import tldextract

# Folders, the link-type codes, and the domain lists the rules use (visual hosts, reddit, messaging).
SAMPLE_DIR = 'sample_998_v4_excl_output'
OUT_DIR = 'link_domain_output'
CODES = ['porn', 'crypto', 'shop', 'blog_channel', 'news', 'media_host', 'messaging', 'social', 'other']
VISUAL_HOSTS = ['i.redd.it', 'v.redd.it', 'preview.redd.it', 'i.reddituploads.com', 'i.redditmedia.com']
VISUAL_DOMAINS = ['imgur.com', 'gfycat.com', 'redgifs.com', 'giphy.com', 'streamable.com', 'youtube.com', 'youtu.be']
REDDIT_DOMAINS = ['reddit.com', 'redd.it', 'redditmedia.com', 'reddituploads.com']   # what is left after the visual hosts
MESSAGING_DOMAINS = ['discord.gg', 'discord.com', 'discordapp.com', 't.me', 'telegram.me', 'telegram.org', 'whatsapp.com', 'wa.me']
# Hand corrections to agent labels: domain -> (link type, date decided, reason).
OVERRIDES = {                                # domain: (link_type, date, why)
    'https': ('other', '2026-10-09', 'malformed URL with no real domain; agent guessed shop from the path'),
}
# A URL inside text, and the offline domain splitter (registered domain = name + public suffix).
URL = re.compile(r'https?://[^\s)\]>"\'<]+', re.I)
ext = tldextract.TLDExtract(suffix_list_urls=())   # offline: bundled public-suffix list

# Write a file only if it does not exist yet (never overwrite); .xlsx goes to Excel, anything else to parquet.
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    if name.endswith('.xlsx'):
        df.to_excel(path, index=False, freeze_panes=(1, 1))
    else:
        df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

## Links before the first accusation

In [ ]:
# Posts and comments the 998 pilot accounts made before their (edit-adjusted) first accusation.
acc = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_accounts.parquet'))[['username_key', 'first_mention_adj_utc']]
p = (pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_posts.parquet'), columns=['id', 'outer_key', 'created_utc', 'url', 'is_self', 'selftext', 'title'])
     .drop_duplicates(['outer_key', 'id']).merge(acc, left_on='outer_key', right_on='username_key'))
p = p[p['created_utc'] < p['first_mention_adj_utc']]
c = (pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_comments.parquet'), columns=['id', 'outer_key', 'created_utc', 'body'])
     .drop_duplicates(['outer_key', 'id']).merge(acc, left_on='outer_key', right_on='username_key'))
c = c[c['created_utc'] < c['first_mention_adj_utc']]

# One row per link: URLs found in post text, the post's own link (link posts only), and URLs in comment text.
recs = []
for r in p.itertuples():
    urls = [('text', u) for u in URL.findall(r.selftext or '')]
    if str(r.is_self) != 'True' and r.url:
        urls.append(('link', r.url))
    recs += [(r.outer_key, r.id, 'post', where, r.created_utc, u) for where, u in urls]
for r in c.itertuples():
    recs += [(r.outer_key, r.id, 'comment', 'text', r.created_utc, u) for u in URL.findall(r.body or '')]
L = pd.DataFrame(recs, columns=['outer_key', 'item_id', 'kind', 'where', 'created_utc', 'url'])
# Host = lowercased hostname without port or www.; domain = registered domain (e.g. old.reddit.com -> reddit.com).
L['host'] = (L['url'].str.extract(r'https?://([^/?#]+)', flags=re.I)[0].str.lower().str.split(':').str[0]
             .str.removeprefix('www.').fillna(''))
L['domain'] = [(e.domain + '.' + e.suffix) if e.suffix else e.domain for e in map(ext, L['host'])]
print('links', len(L), '| accounts', L['outer_key'].nunique(), '| domains', L['domain'].nunique())
L.groupby(['kind', 'where']).size()

## Rules, then agent labels

In [ ]:
# Agent labels for every domain no rule covers; checks that the label set matches this link table exactly.
lab = pd.read_csv(os.path.join(OUT_DIR, 'link_domain_llm_labels_v1.csv'), keep_default_na=False)
ruled = set(VISUAL_DOMAINS + REDDIT_DOMAINS + MESSAGING_DOMAINS)
assert lab['domain'].is_unique and set(lab['domain']) == set(L['domain']) - ruled - {''}, 'agent set differs from this link table'
assert lab['code'].isin(CODES).all()

# Agent label first, then manual overrides, then rules (rules overwrite both).
L['link_type'] = L['domain'].map(lab.set_index('domain')['code'])
L['type_source'] = np.where(L['link_type'].notna(), 'agent', None)
for dom, (t, when, why) in OVERRIDES.items():
    L.loc[L['domain'].eq(dom), ['link_type', 'type_source']] = [t, 'manual']
rule = {}
rule['messaging'] = L['domain'].isin(MESSAGING_DOMAINS)
rule['reddit'] = L['domain'].isin(REDDIT_DOMAINS) | (L['domain'].eq('') & L['url'].str.startswith('/'))   # relative /r/... links
rule['other'] = L['domain'].eq('') & ~L['url'].str.startswith('/')             # broken URLs with no host
rule['visual'] = (L['host'].isin(VISUAL_HOSTS) | L['domain'].isin(VISUAL_DOMAINS)
                  | (L['domain'].eq('reddit.com') & L['url'].str.contains('/gallery/', regex=False)))
for t in ['other', 'messaging', 'reddit', 'visual']:                                      # later wins: a gallery is visual, not reddit
    L.loc[rule[t], ['link_type', 'type_source']] = [t, 'rule']
assert L['link_type'].notna().all()
# Hostless links by first 3 characters, then links, accounts and domains per link type.
print(L.loc[L['domain'].eq(''), 'url'].str[:3].value_counts().to_dict())
pd.DataFrame({'links': L['link_type'].value_counts(), 'accounts': L.groupby('link_type')['outer_key'].nunique(),
              'domains': L.groupby('link_type')['domain'].nunique()}).sort_values('links', ascending=False)

In [ ]:
# agent labels by confidence, and the domains labelled media_host
print(pd.crosstab(lab['code'], lab['confidence']))
media = L[L['link_type'].eq('media_host')].groupby('domain').agg(links=('url', 'size'), accounts=('outer_key', 'nunique'))
print('media_host links', media['links'].sum(), '| accounts', L.loc[L['link_type'].eq('media_host'), 'outer_key'].nunique())
media.sort_values('links', ascending=False)

In [ ]:
# One row per domain: link type(s), link and account counts, top hosts, and the agent's code and reason.
dom = (L.groupby('domain').agg(link_type=('link_type', lambda s: '/'.join(sorted(s.unique()))), links=('url', 'size'),
                               accounts=('outer_key', 'nunique'), hosts=('host', lambda s: ', '.join(s.value_counts().index[:3])))
       .join(lab.set_index('domain')[['code', 'confidence', 'reason']].rename(columns={'code': 'agent_code'}))
       .reset_index().sort_values('links', ascending=False))
# Save the link table, the domain table, and the domain table as a review sheet.
save_new(L, 'links_pre_accusation_v1.parquet')
save_new(dom, 'link_domains_v1.parquet')
save_new(dom, 'link_domains_v1_review.xlsx')